# Zipformer code-switch (VI+EN) fine-tuning on Colab

End-to-end on Colab + Google Drive:
**setup -> select VI+EN utterances -> materialize combined dataset -> compute fbank -> resumable fine-tune.**

Reuses `compute_fbank_huggingface.py` and `finetune.py` from this repo. Heavy
artifacts (cuts/feats + checkpoints) live on Drive so they survive runtime
restarts. You must place **your own** `bpe.model` and `pretrained.pt` on Drive.

Run cells top-to-bottom. Use a CPU runtime for selection/materialize/fbank and a
GPU runtime for fine-tuning.

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# --- EDIT THESE -----------------------------------------------------------
REPO_URL   = ''  # optional: git URL of THIS repo. Leave '' if you upload it.
REPO       = '/content/asr_zipformer'                 # where this repo lives
DRIVE      = '/content/drive/MyDrive/vibank_cs'       # persistent workspace
PREFIX     = 'vibank_cs'
BPE_MODEL  = f'{DRIVE}/bpe.model'        # <- upload YOUR bpe.model here
PRETRAINED = f'{DRIVE}/pretrained.pt'    # <- upload YOUR checkpoint here
TEXT_NORM  = 'none'   # MUST match how your BPE was trained (see BPE check)
# -------------------------------------------------------------------------

ASR      = f'{REPO}/icefall/egs/librispeech/ASR'
FBANK    = f'{DRIVE}/data/fbank'
EXP_DIR  = f'{DRIVE}/exp_finetune'
COMBINED = f'{DRIVE}/combined_dataset'
MANIFEST = f'{DRIVE}/combined_manifest.jsonl'
os.makedirs(DRIVE, exist_ok=True)

if REPO_URL and not os.path.isdir(REPO):
    !git clone {REPO_URL} {REPO}
assert os.path.isdir(REPO), f'Put this repo at {REPO} (clone or upload).'
print('REPO  =', REPO)
print('DRIVE =', DRIVE)

In [ ]:
# Python deps. k2 is the fragile one: it must match Colab's torch + CUDA.
import torch
print('torch:', torch.__version__, 'cuda:', torch.version.cuda)

!pip -q install lhotse datasets soundfile sentencepiece wordfreq kaldialign num2words
# k2: pick the wheel matching the torch/cuda printed above. The -f index lists
# all builds; if the bare install fails, copy the exact matching URL from there.
!pip -q install k2 -f https://k2-fsa.github.io/k2/cuda.html || \
  echo 'k2 install failed - open https://k2-fsa.github.io/k2/cuda.html and pick the wheel for the torch/cuda above'
!pip -q install -e {REPO}/icefall

import k2, lhotse, datasets
print('k2:', k2.__version__, '| lhotse:', lhotse.__version__, '| datasets:', datasets.__version__)

In [ ]:
# Place the repo's maintained scripts where the recipe expects them.
import shutil
shutil.copy(f'{REPO}/compute_fbank_huggingface.py', f'{ASR}/local/compute_fbank_huggingface.py')
shutil.copy(f'{REPO}/finetune.py', f'{ASR}/zipformer/finetune.py')
print('scripts placed under', ASR)

## 1. Filter sanity check

In [ ]:
%cd {REPO}/colab
!python test_codeswitch_filter.py

## 2. Select VI+EN utterances

Edit `sources` with your candidate datasets. `text_key`/`audio_key` are the
column names in each dataset. Start with `--limit` for a dry run, review the
samples, tune, then drop `--limit` for the full scan.

In [ ]:
import json
sources = [
    {'dataset': 'org/your-vi-dataset-1', 'config': None, 'split': 'train',
     'text_key': 'transcription', 'audio_key': 'audio'},
    # {'dataset': 'org/your-vi-dataset-2', 'split': 'train', 'text_key': 'sentence'},
]
with open('sources.json', 'w', encoding='utf-8') as f:
    json.dump(sources, f, ensure_ascii=False, indent=2)
print(open('sources.json', encoding='utf-8').read())

In [ ]:
# Dry run: scan up to 5000 rows/source. Remove --limit for the real selection.
!python select_codeswitch_utterances.py \
  --sources-json sources.json \
  --output-dir {DRIVE} \
  --min-en-tokens 1 \
  --limit 5000

In [ ]:
# Review precision before scaling. Tune --min-en-tokens / --max-en-ratio /
# wordlists if too many false positives appear here.
print('=== SELECTED (english tokens shown) ===')
print(open(f'{DRIVE}/samples_selected.txt', encoding='utf-8').read()[:4000])
print('\n=== REJECTED ===')
print(open(f'{DRIVE}/samples_rejected.txt', encoding='utf-8').read()[:2000])
print('\n=== STATS ===')
print(open(f'{DRIVE}/selection_stats.json', encoding='utf-8').read())

## 3. BPE coverage check (BLOCKING)

Confirm your BPE tokenizes the selected (normalized) transcripts without an
`<unk>` flood and that `TEXT_NORM` round-trips Vietnamese correctly. If this
looks wrong, fix `TEXT_NORM` (likely `'none'`) before computing any features.

In [ ]:
import json, sentencepiece as spm, sys
sys.path.insert(0, f'{ASR}/local')
from compute_fbank_huggingface import normalize_text

sp = spm.SentencePieceProcessor(); sp.load(BPE_MODEL)
unk = sp.piece_to_id('<unk>')
rows = [json.loads(l) for l in open(MANIFEST, encoding='utf-8')][:20]
tot = unkn = 0
for r in rows:
    norm = normalize_text(r['text'], TEXT_NORM)
    ids = sp.encode(norm, out_type=int)
    tot += len(ids); unkn += sum(i == unk for i in ids)
    print(f"{norm}\n  -> {sp.encode(norm, out_type=str)}\n")
print(f'tokens={tot} unk={unkn} ({100*unkn/max(tot,1):.1f}%)')
assert tot == 0 or unkn / tot < 0.02, 'Too many <unk>; check TEXT_NORM / BPE.'

## 4. Materialize combined dataset to Drive

In [ ]:
!python materialize_dataset.py \
  --manifest {MANIFEST} \
  --output-dir {COMBINED} \
  --sampling-rate 16000 \
  --dev-frac 0.02

## 5. Compute fbank (train perturbed, dev clean) -> Drive

`--load-from-disk` reads the combined dataset; `--split train|dev` selects the
DatasetDict split saved above.

In [ ]:
%cd {ASR}
import os; os.makedirs(FBANK, exist_ok=True)

# Train (3x speed perturbation).
!python local/compute_fbank_huggingface.py \
  --load-from-disk --dataset {COMBINED} --split train \
  --output-name train --prefix {PREFIX} --output-dir {FBANK} \
  --text-normalization {TEXT_NORM} --resample-rate 16000 \
  --bpe-model {BPE_MODEL} --perturb-speed

# Dev (no perturbation).
!python local/compute_fbank_huggingface.py \
  --load-from-disk --dataset {COMBINED} --split dev \
  --output-name dev --prefix {PREFIX} --output-dir {FBANK} \
  --text-normalization {TEXT_NORM} --resample-rate 16000 \
  --bpe-model {BPE_MODEL}

In [ ]:
# Duration distribution sanity check.
!python local/display_manifest_statistics.py 2>/dev/null || \
  python -c "from lhotse import load_manifest_lazy; \
cs=load_manifest_lazy('{FBANK}/{PREFIX}_cuts_train.jsonl.gz'); \
import itertools; print('train cuts:', len(list(itertools.islice((c for c in cs),0,10**9))))"

## 6. Fine-tune (GPU, resumable)

Switch the runtime to **GPU** first. Checkpoints write to `EXP_DIR` on Drive.
Re-running a session with a higher `--start-epoch` resumes from Drive.

In [ ]:
# Smoke test: 1 epoch, tiny batch. Confirms ckpt loads + a checkpoint is written.
%cd {ASR}
!python zipformer/finetune.py \
  --world-size 1 --num-epochs 1 --start-epoch 1 --use-fp16 1 \
  --do-finetune 1 --finetune-ckpt {PRETRAINED} --base-lr 0.0045 --use-mux 0 \
  --bpe-model {BPE_MODEL} --exp-dir {EXP_DIR} \
  --manifest-dir {FBANK} \
  --train-manifest {PREFIX}_cuts_train.jsonl.gz \
  --valid-manifest {PREFIX}_cuts_dev.jsonl.gz \
  --valid-set-name {PREFIX} \
  --enable-musan 0 --max-duration 80

In [ ]:
# Full run. To RESUME in a later session: set --start-epoch N (loads
# epoch-{N-1}.pt from Drive) and --do-finetune 0.
%cd {ASR}
!python zipformer/finetune.py \
  --world-size 1 --num-epochs 20 --start-epoch 1 --use-fp16 1 \
  --do-finetune 1 --finetune-ckpt {PRETRAINED} --base-lr 0.0045 --use-mux 0 \
  --bpe-model {BPE_MODEL} --exp-dir {EXP_DIR} \
  --manifest-dir {FBANK} \
  --train-manifest {PREFIX}_cuts_train.jsonl.gz \
  --valid-manifest {PREFIX}_cuts_dev.jsonl.gz \
  --valid-set-name {PREFIX} \
  --enable-musan 0 --max-duration 300